## **IDN**

In [1]:
import os, re, json, math, pdfplumber, torch, csv
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "facebook/bart-large-mnli"
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
label2id   = {v.lower(): k for k, v in model.config.id2label.items()}


In [ ]:
import re
import csv
import json
import torch
import pdfplumber
from tqdm import tqdm
from pathlib import Path

def extract_pdf(path: str) -> str:
    text = []
    with pdfplumber.open(path) as pdf:
        for p in pdf.pages:
            page = p.extract_text() or ""
            text.append(page)
    return re.sub(r"\s+", " ", " ".join(text)).strip()


def chunk_text(txt: str, max_tokens: int = 256, overlap: int = 64):
    words = txt.split()
    step = max_tokens - overlap
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + max_tokens])


@torch.inference_mode()
def nli_score(premise_chunks, hypothesis: str):
    max_scores = {"entailment": 0.0, "neutral": 0.0, "contradiction": 0.0}
    for prem in premise_chunks:
        enc = tokenizer(prem, hypothesis, truncation=True, return_tensors="pt").to(device)
        logits = model(**enc).logits
        probs = torch.softmax(logits, -1)[0]
        max_scores["entailment"] = max(max_scores["entailment"], probs[label2id["entailment"]].item())
        max_scores["neutral"] = max(max_scores["neutral"], probs[label2id["neutral"]].item())
        max_scores["contradiction"] = max(max_scores["contradiction"], probs[label2id["contradiction"]].item())
    return max_scores


def extract_range(filename):
    match = re.search(r"(\d{1,3}-\d{1,3})", filename)
    return match.group(1) if match else None


def process_all(doc_dir: str, json_dir: str, output_score: str):
    doc_dir = Path(doc_dir)
    json_dir = Path(json_dir)
    output_score = Path(output_score)
    output_score.mkdir(parents=True, exist_ok=True)

    # Loop per subdirektori (contoh D1-xxx)
    for doc_subdir in sorted(doc_dir.iterdir()):
        if not doc_subdir.is_dir():
            continue

        dir_prefix = doc_subdir.name.split("-")[0]  # contoh: D3
        json_subdir = next((d for d in json_dir.iterdir() if d.name.startswith(dir_prefix)), None)

        if not json_subdir:
            print(f"Skip: Tidak ditemukan pasangan untuk {doc_subdir.name}!!")
            continue

        print(f"\n=== Memproses direktori: {doc_subdir.name} ===")

        results = []
        pdf_files = list(doc_subdir.glob("*.pdf"))
        json_files = list(json_subdir.glob("*.json"))

        json_map = {extract_range(f.name): f for f in json_files if extract_range(f.name)}

        for pdf_path in tqdm(pdf_files, desc=f"Proses {doc_subdir.name}"):
            range_key = extract_range(pdf_path.name)
            if not range_key or range_key not in json_map:
                print(f" Skip: {pdf_path.name} tidak ada pasangan JSON yang cocok")
                continue

            json_path = json_map[range_key]
            try:
                premise = extract_pdf(str(pdf_path))
                premise_chunks = list(chunk_text(premise))

                qna_data = json.loads(json_path.read_text(encoding="utf-8"))
                source = "grok" if "grok" in json_path.name.lower() else "gpt"

                for item in qna_data:
                    question = item.get("Question", "").strip()
                    answer = item.get("Answer", "").strip()
                    hypothesis = answer or f"{question} {answer}".strip()
                    scores = nli_score(premise_chunks, hypothesis)

                    results.append({
                        "file": pdf_path.stem,
                        "generator": source,
                        "question": question,
                        "answer": answer,
                        "score_entailment": round(scores["entailment"], 4),
                        "score_neutral": round(scores["neutral"], 4),
                        "score_contradiction": round(scores["contradiction"], 4),
                        "score_final": round(scores["entailment"] - scores["contradiction"], 4),
                    })
            except Exception as e:
                print(f"Gagal memproses {pdf_path.name}: {e}")

        output_csv = output_score / f"{doc_subdir.name}.csv"
        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "file", "generator", "question", "answer",
                "score_entailment", "score_neutral", "score_contradiction", "score_final"
            ], delimiter=";")
            writer.writeheader()
            writer.writerows(results)

        print(f"Selesai: {doc_subdir.name} → {output_csv}")


if __name__ == "__main__":
    process_all(
        doc_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dokumen\idn",
        json_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dataset\idn",
        output_score=r"C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn2"
    )


## **IDN OCR**

In [1]:
import os, re, json, math, pdfplumber, torch, csv
import pytesseract
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

MODEL_NAME = "facebook/bart-large-mnli"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
label2id = {v.lower(): k for k, v in model.config.id2label.items()}

In [ ]:
import re
import csv
import json
import torch
import pytesseract
from tqdm import tqdm
from pathlib import Path
from pdf2image import convert_from_path

def extract_text_from_scanned_pdf(pdf_path: str) -> str:
    try:
        images = convert_from_path(pdf_path, dpi=300)
        text_list = [pytesseract.image_to_string(img, lang="ind") for img in images]
        return re.sub(r"\s+", " ", "\n".join(text_list)).strip()
    except Exception as e:
        print(f"OCR gagal untuk {pdf_path}: {e}")
        return ""

def chunk_text(txt: str, max_tokens: int = 256, overlap: int = 64):
    words = txt.split()
    step = max_tokens - overlap
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + max_tokens])

@torch.inference_mode()
def nli_score(premise_chunks, hypothesis: str):
    max_scores = {"entailment": 0.0, "neutral": 0.0, "contradiction": 0.0}
    for prem in premise_chunks:
        enc = tokenizer(prem, hypothesis, truncation=True, return_tensors="pt").to(device)
        logits = model(**enc).logits
        probs = torch.softmax(logits, -1)[0]
        for label in max_scores:
            max_scores[label] = max(max_scores[label], probs[label2id[label]].item())
    return max_scores


def extract_range(filename):
    match = re.search(r"(\d{1,3}-\d{1,3})", filename)
    return match.group(1) if match else None


# === Main Otomatisasi Semua Folder ===
def process_all(main_doc_dir: str, main_json_dir: str, output_root: str):
    main_doc_dir = Path(main_doc_dir)
    main_json_dir = Path(main_json_dir)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    # Loop per subdirektori dokumen
    for doc_subdir in sorted(main_doc_dir.iterdir()):
        if not doc_subdir.is_dir():
            continue

        dir_prefix = doc_subdir.name.split("-")[0]  # contoh: D3
        json_subdir = next((d for d in main_json_dir.iterdir() if d.name.startswith(dir_prefix)), None)

        if not json_subdir:
            print(f"Skip: Tidak ditemukan pasangan folder dataset untuk {doc_subdir.name}")
            continue

        print(f"\n=== Memproses direktori: {doc_subdir.name} ===")

        results = []
        pdf_files = list(doc_subdir.glob("*.pdf"))
        json_files = list(json_subdir.glob("*.json"))

        json_map = {extract_range(f.name): f for f in json_files if extract_range(f.name)}

        for pdf_path in tqdm(pdf_files, desc=f"Proses {doc_subdir.name}"):
            range_key = extract_range(pdf_path.name)
            if not range_key or range_key not in json_map:
                print(f"Skip: {pdf_path.name} tidak ada pasangan JSON cocok.")
                continue

            json_path = json_map[range_key]
            try:
                teks_ocr = extract_text_from_scanned_pdf(str(pdf_path))
                premise_chunks = list(chunk_text(teks_ocr))
                qna_data = json.loads(json_path.read_text(encoding="utf-8"))
                source = "grok" if "grok" in json_path.name.lower() else "gpt"

                for item in qna_data:
                    question = item.get("Question", "").strip()
                    answer = item.get("Answer", "").strip()
                    hypothesis = answer or f"{question} {answer}".strip()
                    scores = nli_score(premise_chunks, hypothesis)

                    results.append({
                        "file": pdf_path.stem,
                        "generator": source,
                        "question": question,
                        "answer": answer,
                        "score_entailment": round(scores["entailment"], 4),
                        "score_neutral": round(scores["neutral"], 4),
                        "score_contradiction": round(scores["contradiction"], 4),
                        "score_final": round(scores["entailment"] - scores["contradiction"], 4)
                    })
            except Exception as e:
                print(f"Gagal memproses {pdf_path.name}: {e}")

        output_csv = output_root / f"{doc_subdir.name}.csv"
        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "file", "generator", "question", "answer",
                "score_entailment", "score_neutral", "score_contradiction", "score_final"
            ], delimiter=";")
            writer.writeheader()
            writer.writerows(results)

        print(f"Selesai: {doc_subdir.name} → {output_csv}")


if __name__ == "__main__":
    pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    process_all(
        main_doc_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dokumen\idn-ocr",
        main_json_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dataset\idn-ocr",
        output_root=r"C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\idn-ocr2"
    )


## **ENG**

In [ ]:
import os, re, json, csv, math, torch, pytesseract, nltk
from tqdm import tqdm
from pathlib import Path
from pdf2image import convert_from_path
from nltk.tokenize import sent_tokenize
from transformers import (MarianTokenizer, MarianMTModel, AutoTokenizer, AutoModelForSequenceClassification)

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
nltk.download('punkt')
nltk.download('punkt_tab')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Load Translation Model EN-ID ===
trans_model_name = "facebook/nllb-200-distilled-600M"
trans_tokenizer = MarianTokenizer.from_pretrained(trans_model_name)
trans_model = MarianMTModel.from_pretrained(trans_model_name).to(device)

# === Load NLI Model ===
nli_model_name = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)
label2id = {v.lower(): k for k, v in nli_model.config.id2label.items()}


In [ ]:
import os
import re
import csv
import json
import torch
from tqdm import tqdm
from pathlib import Path
from pdf2image import convert_from_path
import pytesseract
from nltk.tokenize import sent_tokenize

# === OCR (ENG) ===
def extract_text_from_scanned_pdf(pdf_path: str) -> str:
    print(f"OCR dari: {pdf_path}")
    images = convert_from_path(pdf_path, dpi=300)
    text_list = [pytesseract.image_to_string(img, lang="eng") for img in images]
    return re.sub(r"\s+", " ", "\n".join(text_list)).strip()


# === Translate ENG -> IDN ===
def translate_text(text: str) -> str:
    sentences = sent_tokenize(text)
    translated_sentences = []

    for sentence in tqdm(sentences, desc="Translasi"):
        inputs = trans_tokenizer(sentence, return_tensors="pt", truncation=True, padding=True).to(device)
        outputs = trans_model.generate(**inputs, max_length=512)
        translated = trans_tokenizer.decode(outputs[0], skip_special_tokens=True)
        translated_sentences.append(translated)

    return " ".join(translated_sentences)


# === Potong teks ===
def chunk_text(txt: str, max_tokens: int = 256, overlap: int = 64):
    words = txt.split()
    step = max_tokens - overlap
    for i in range(0, len(words), step):
        yield " ".join(words[i:i + max_tokens])


# === Hitung skor NLI (entailment, neutral, contradiction) ===
@torch.inference_mode()
def nli_score(premise_chunks, hypothesis: str):
    max_scores = {"entailment": 0.0, "neutral": 0.0, "contradiction": 0.0}
    for prem in premise_chunks:
        enc = nli_tokenizer(prem, hypothesis, truncation=True, return_tensors="pt").to(device)
        logits = nli_model(**enc).logits
        probs = torch.softmax(logits, -1)[0]
        for label in max_scores:
            max_scores[label] = max(max_scores[label], probs[label2id[label]].item())
    return {k: round(v, 4) for k, v in max_scores.items()}


def process_all(base_pdf_dir: str, base_json_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    doc_dirs = [d for d in Path(base_pdf_dir).iterdir() if d.is_dir()]

    for doc_dir in doc_dirs:
        dir_name = doc_dir.name
        json_dir = Path(base_json_dir) / dir_name
        if not json_dir.exists():
            print(f"Tidak ditemukan pasangan dataset untuk {dir_name}")
            continue

        print(f"\n=== Evaluasi direktori: {dir_name} ===")
        results = []

        pdf_files = list(doc_dir.glob("*.pdf"))
        json_files = list(json_dir.glob("*.json"))

        def extract_range(filename):
            match = re.search(r"(\d{1,3}-\d{1,3})", filename)
            return match.group(1) if match else None

        json_map = {extract_range(f.name): f for f in json_files if extract_range(f.name)}

        with tqdm(total=len(pdf_files), desc=f"Evaluasi {dir_name}") as pbar:
            for pdf_path in pdf_files:
                range_key = extract_range(pdf_path.name)
                if not range_key or range_key not in json_map:
                    tqdm.write(f"Tidak menemukan JSON untuk {pdf_path.name}")
                    pbar.update(1)
                    continue

                json_path = json_map[range_key]
                source = "grok" if "grok" in json_path.name.lower() else "gpt"
                tqdm.write(f"\nProses: {pdf_path.name} + {json_path.name}")

                # === OCR + Translate ===
                ocr_text = extract_text_from_scanned_pdf(str(pdf_path))
                id_text = translate_text(ocr_text)
                chunks = list(chunk_text(id_text))

                # === Load dataset JSON ===
                with open(json_path, "r", encoding="utf-8") as f:
                    qna_data = json.load(f)

                # === Evaluasi tiap QnA ===
                for item in qna_data:
                    question = item.get("Question", "").strip()
                    answer = item.get("Answer", "").strip()
                    hypothesis = answer or f"{question} {answer}".strip()

                    scores = nli_score(chunks, hypothesis)

                    results.append({
                        "file": pdf_path.stem,
                        "generator": source,
                        "question": question,
                        "answer": answer,
                        "score_entailment": scores["entailment"],
                        "score_neutral": scores["neutral"],
                        "score_contradiction": scores["contradiction"],
                        "score_final": round(scores["entailment"] - scores["contradiction"], 4)
                    })

                pbar.update(1)

        output_csv = Path(output_dir) / f"{dir_name}.csv"
        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=[
               "file", "generator", "question", "answer",
                "score_entailment", "score_neutral", "score_contradiction", "score_final"
            ], delimiter=";")
            writer.writeheader()
            writer.writerows(results)

        print(f"Selesai: {output_csv}")


if __name__ == "__main__":
    process_all(
        base_pdf_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dokumen\eng",
        base_json_dir=r"C:\Users\Liza\Documents\Kerja Praktik\dataset\eng",
        output_dir=r"C:\Users\Liza\Documents\Kerja Praktik\NLI\Hasil score\eng2"
    )
